# Emotion Vectors — ストーリー生成 (Colaboratory版)

論文「Emotion Concepts and their Function in a Large Language Model」(Anthropic, 2026) の再現実装。  
Gemma 2 2B-it で感情ストーリーを生成し、Google Drive に保存する。

## 担当範囲
- 感情: emotions_full.json の 167〜170番（weary, worn out, worried, worthless）
- トピック: 100件（論文フル）
- ストーリー数/トピック: 12件（論文フル）

## 実行手順
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. 左サイドバー 🔑 シークレット → `HF_TOKEN` に HuggingFace トークンを登録
3. 全セルを順番に実行

---

In [ ]:
# Cell 1: GPU確認
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU が見つかりません。\n"
        "ランタイム → ランタイムのタイプを変更 → T4 GPU を選択してください。"
    )

props = torch.cuda.get_device_properties(0)
print(f"GPU  : {props.name}")
print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

In [ ]:
# Cell 2: パッケージインストール
!pip install -q 'transformers>=4.40' accelerate tqdm huggingface_hub

In [ ]:
# Cell 3: Google Drive マウント & ディレクトリ設定
from google.colab import drive
from pathlib import Path
import json, os

drive.mount('/content/drive')

# Drive上の保存先（セッションが切れても残る）
DRIVE_BASE  = Path('/content/drive/MyDrive/emotion-vectors')
STORIES_DIR = DRIVE_BASE / 'stories'
STORIES_DIR.mkdir(parents=True, exist_ok=True)

existing = list(STORIES_DIR.rglob('*.json'))
print(f"保存先      : {STORIES_DIR}")
print(f"既存ファイル: {len(existing)} 件")

In [ ]:
# Cell 4: HuggingFace ログイン & 実験設定
from google.colab import userdata
from huggingface_hub import login

try:
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print("HuggingFace ログイン成功")
except Exception as e:
    print(f"警告: {e}")
    print("Gemma 2 はアクセス許可が必要です。HF_TOKEN を Colab シークレットに登録してください。")

# 担当感情: emotions_full.json の 167〜170番
EMOTIONS = ["weary", "worn out", "worried", "worthless"]

# トピック: 論文フル100件
TOPICS = [
    "An artist discovers someone has tattooed their work",
    "A family member announces they're converting to a different religion",
    "Someone's childhood imaginary friend appears in their niece's drawings",
    "A person finds out their biography was written without their knowledge",
    "A neighbor starts a renovation project",
    "Someone finds their grandmother's engagement ring in a pawn shop",
    "A student learns their scholarship application was denied",
    "A person's online friend turns out to live in the same city",
    "A neighbor wants to install a fence",
    "An adult child moves back in with their parents",
    "An employee is asked to train their replacement",
    "An athlete is asked to switch positions",
    "A traveler's flight is delayed, causing them to miss an important event",
    "A student is accused of plagiarism",
    "A person discovers their mentor has retired without saying goodbye",
    "Two friends both apply for the same job",
    "A person runs into their ex at a mutual friend's wedding",
    "Someone discovers their friend has been lying about their job",
    "A person discovers their partner has been taking secret phone calls",
    "A person discovers their child has the same teacher they had",
    "A person's car is towed from their own driveway",
    "Two friends realize they remember a shared event completely differently",
    "Someone discovers their mother kept every school assignment",
    "A person discovers their teenage diary has been published online",
    "Someone finds out their medical records were mixed up with another patient's",
    "A person finds out their article was published under someone else's name",
    "An athlete doesn't make the team they expected to join",
    "An employee is transferred to a different department",
    "Someone receives a friend request from a childhood bully",
    "A person finds out their surprise party has been cancelled",
    "An employee finds out a junior colleague makes more money",
    "A person finds out their partner has been learning their native language",
    "A chef receives a harsh review from a food critic",
    "A person learns their favorite restaurant is closing",
    "Someone finds their childhood teddy bear at a yard sale",
    "A homeowner discovers previous residents left items in the attic",
    "Someone finds an unsigned birthday card in their mailbox",
    "Someone discovers a hidden room in their new house",
    "Two strangers realize they've been dating the same person",
    "A person finds a hidden letter in a used book",
    "Two siblings inherit their grandmother's house",
    "Someone finds a wallet containing a large sum of cash",
    "Someone receives an invitation to their high school reunion",
    "Someone discovers their recipe has become famous under another name",
    "A college student discovers their roommate has been reading their journal",
    "A person finds out they were adopted through a DNA test",
    "A family member wants to sell a cherished heirloom",
    "Someone receives a package intended for the previous tenant",
    "Someone's childhood home is about to be demolished",
    "A person's invention is already patented by someone else",
    "A neighbor's dog keeps escaping into their yard",
    "A coach has to cut a player from the team",
    "Someone learns their favorite author plagiarized their stories",
    "A student finds out their scholarship was meant for someone else",
    "Someone discovers their teenager has a secret social media account",
    "Two roommates disagree about getting a pet",
    "Two friends plan separate birthday parties on the same day",
    "A person learns their childhood best friend doesn't remember them",
    "A musician hears their song being performed by someone else",
    "A person's manuscript is rejected by their dream publisher",
    "A person finds old photos that contradict family stories",
    "A person is asked to give a speech at their parent's retirement party",
    "A student discovers their teacher follows them on social media",
    "A parent finds an old letter they wrote but never sent",
    "An employee discovers the company is being sold",
    "A person accidentally sends a text to the wrong recipient",
    "Two coworkers are stuck in an elevator for three hours",
    "A student learns their thesis advisor is leaving the university",
    "A person's longtime hobby becomes their child's obsession",
    "Two colleagues are both considered for the same promotion",
    "Two coworkers discover they went to the same summer camp",
    "A tenant receives an eviction notice",
    "Someone finds their parent's draft letter of resignation from decades ago",
    "Someone finds out their best friend is moving across the country",
    "A neighbor's tree falls on their property",
    "Someone receives an apology letter years after the incident",
    "A person discovers the tree they planted as a child has been cut down",
    "Two siblings discover different versions of their inheritance",
    "A person finds their childhood home listed for sale online",
    "A homeowner learns their house was a former crime scene",
    "Someone finds out they have a half-sibling they never knew about",
    "A person learns their childhood bully became a therapist",
    "Two people discover they've been working on identical projects",
    "A person finds their spouse's secret savings account",
    "A neighbor complains about noise levels",
    "Someone finds their deceased parent's bucket list",
    "A teacher receives an unexpected gift from a former student",
    "An artist's work is displayed without their permission",
    "Someone discovers their neighbor is secretly wealthy",
    "A student receives a much lower grade than expected",
    "A person learns their college is closing down",
    "A neighbor asks to cut down a tree on the property line",
    "Two strangers discover they share the same rare medical condition",
    "Someone receives flowers with no card attached",
    "Someone discovers their partner has been writing a novel about them",
    "Someone finds a time capsule they don't remember burying",
    "Someone finds their partner's bucket list",
    "A neighbor asks to use part of the yard for a garden",
    "A person learns their apartment building is going condo",
    "Someone finds their college application essay published as an example",
]

N_STORIES_PER_TOPIC = 12
print(f"担当感情         : {EMOTIONS}")
print(f"感情数           : {len(EMOTIONS)}")
print(f"トピック数       : {len(TOPICS)}")
print(f"目標ストーリー数 : {len(EMOTIONS) * len(TOPICS) * N_STORIES_PER_TOPIC}")

In [ ]:
# Cell 5: モデルロード
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "google/gemma-2-2b-it"

print(f"ロード中: {MODEL_NAME} ...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"ロード完了 : {time.time() - t0:.1f}s")
print(f"VRAM使用  : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 6: ストーリー生成関数
import re

STORY_PROMPT = """\
Write {n} different stories based on the following premise.

Topic: {topic}

The story should follow a character who is feeling {emotion}.

Format the stories like so:

[story 1]

[story 2]

[story 3]

etc.

The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.

IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:

- The character's actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions

The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.\
"""


def split_stories(raw: str, n: int, min_chars: int = 150) -> list:
    parts = re.split(
        r"(?:\[story\s*\d+\]"
        r"|\*\*?#{1,3}\s+[^\n]+"
        r"|#{2,3}\s+[^\n]+"
        r"|\*\*Story\s*\d+\*\*"
        r"|\*{3,}|---+)",
        raw, flags=re.IGNORECASE,
    )
    return [p.strip() for p in parts if len(p.strip()) >= min_chars][:n]


def generate_stories(emotion: str, topic: str, n: int) -> list:
    prompt = STORY_PROMPT.format(n=n, topic=topic, emotion=emotion)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return split_stories(raw, n)


print("関数定義完了")

In [ ]:
# Cell 7: ストーリー生成メインループ
# セッションが切れても再実行すれば続きから再開される（チェックポイント済みファイルはスキップ）
from tqdm.notebook import tqdm

t_start = time.time()
done = skipped = failed = 0

tasks = [(e, i, t) for e in EMOTIONS for i, t in enumerate(TOPICS)]

with tqdm(total=len(tasks), desc="生成中") as pbar:
    for emotion, topic_idx, topic in tasks:
        emo_dir = STORIES_DIR / emotion
        emo_dir.mkdir(exist_ok=True)
        out_path = emo_dir / f"topic{topic_idx:02d}.json"

        pbar.set_postfix(emotion=emotion[:8], topic=topic_idx)

        if out_path.exists():
            skipped += 1
            done += 1
            pbar.update(1)
            continue

        try:
            stories = generate_stories(emotion, topic, N_STORIES_PER_TOPIC)
            out_path.write_text(json.dumps({
                "emotion": emotion,
                "topic": topic,
                "topic_idx": topic_idx,
                "stories": stories,
            }, ensure_ascii=False, indent=2))
            done += 1
        except Exception as ex:
            print(f"\n[ERROR] {emotion}/topic{topic_idx:02d}: {ex}")
            failed += 1
            done += 1

        pbar.update(1)

elapsed = time.time() - t_start
print(f"\n生成: {done - skipped - failed}件  スキップ: {skipped}件  エラー: {failed}件")
print(f"経過: {elapsed/60:.1f}分")

In [ ]:
# Cell 8: 進捗確認
total_target = len(EMOTIONS) * len(TOPICS)
completed = list(STORIES_DIR.rglob('*.json'))
print(f"完了: {len(completed)} / {total_target} ({len(completed)/total_target*100:.1f}%)")
print()
for emotion in EMOTIONS:
    emo_dir = STORIES_DIR / emotion
    count = len(list(emo_dir.glob('*.json'))) if emo_dir.exists() else 0
    bar = '█' * count + '░' * (len(TOPICS) - count)
    print(f"  {emotion:14s} [{bar}] {count}/{len(TOPICS)}")